完整的 MinGPT 函数，多头注意力

相比单头注意力，新增：

H=2 #多头注意力的头数，需要满足D%H==0

Wo=torch.randn(D,D,requires_grad=True) #多头输出投影矩阵

MultiHeadCausalSelfAttention里面的修改，拆成多头，计算后合并多头

In [40]:
import numpy as np
import torch
import math
import torch.nn.functional as F

T_max=128 #预设的最大序列长度
D=4 #隐藏维度
V=26 #词表大小
H=2 #都头注意力的头数，需要满足D%H==0
char_to_idx={} #字符到整数索引的映射,形状：{V}（V 是字符表大小）
idx_to_char={} #整数索引回字符的映射,形状：{V}


for i in range(26):
    char_to_idx[chr(ord('a')+i)]=i
    idx_to_char[i]=chr(ord('a')+i)

In [41]:
embedding_matrix=torch.empty(26,D,requires_grad=True) #向量化，先随便填的。
torch.nn.init.normal_(embedding_matrix, mean=0.0, std=0.02)

def TokenEmbedding(token_ids): #输入：token_ids [T] → 输出：x [T, D]（D 是隐藏维度）。或者输入：token_ids [B, T] → 输出：x [B, T, D]。高级索引机制，就是好用。另外这里没有包含注意力机制的Encoder Block。这个代码是decoder-only的。
    return embedding_matrix[token_ids] #向量化。注意不能写成torch.tensor(embedding_matrix[token_ids])，这是因为embedding_matrix[token_ids] 本身已经是一个 tensor 了，再用 torch.tensor() 包裹它，相当于创建了一个全新的、不带梯度历史的 tensor。

In [42]:
pos_embedding_matrix=torch.randn(T_max,D,requires_grad=True) #位置编码变量,形状[T_max, D]
torch.nn.init.normal_(pos_embedding_matrix, mean=0.0, std=0.02)

def LearnablePositionalEncoding(X): #先不用 RoPE，用简单可学习位置编码；输入：x [B, T, D] → 输出：x + pos_embedding_matrix [B, T, D]
    return X+pos_embedding_matrix[:X.size(-2),:].unsqueeze(0) #.unsqueeze(0) 是 PyTorch 中用于增加张量维度的函数，具体作用是在第 0 维（即最前面）插入一个大小

In [43]:
Wq=torch.randn(D,D,requires_grad=True) #生成 Q/K/V 的三个权重矩阵，每个形状：[D, D]。没训练前先随机
Wk=torch.randn(D,D,requires_grad=True)
Wv=torch.randn(D,D,requires_grad=True)
Wo=torch.randn(D,D,requires_grad=True) #多头输出投影矩阵

torch.nn.init.normal_(Wq, mean=0.0, std=0.02)
torch.nn.init.normal_(Wk, mean=0.0, std=0.02)
torch.nn.init.normal_(Wv, mean=0.0, std=0.02)
torch.nn.init.normal_(Wo, mean=0.0, std=0.02)

def MultiHeadCausalSelfAttention(X): #输入：x [B, T, D] → 输出：attn_output [B, T, D]
    B, T, D=X.shape
    Dh=D//H #每个头的维度

    Q=torch.matmul(X,Wq) #直接乘是没问题的，pytorch支持。
    K=torch.matmul(X,Wk)
    V=torch.matmul(X,Wv)
    Q=Q.view(B, T, H, Dh).permute(0, 2, 1, 3)#这一步是拆分多头，从 [B, T, D] 拆成 [B, T, H, Dh]，再把头维度 H 提前到 [B, H, T, Dh]。.permute(0, 2, 1, 3)：4 维张量中，保持第 0、3 维不变，交换第 1 维和第 2 维的位置。
    K=K.view(B, T, H, Dh).permute(0, 2, 1, 3)
    V=V.view(B, T, H, Dh).permute(0, 2, 1, 3)

    S=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(Dh) #原始注意力分数 [B, H, T, T]
    

    M=torch.zeros((T,T),dtype=torch.float32)
    rows,cols=torch.triu_indices(T,T,offset=1) #矩阵的上三角，offset=1表示主对角右移1
    M[rows,cols]=-torch.inf #将上三角位置设为负无穷
    S=S+M #注意力掩码处理后的分数[B, H, T, T]

    exp_S=torch.exp(S)
    A=exp_S/torch.sum(exp_S,axis=-1,keepdims=True) #注意力权重[B, H, T, T]

    O=torch.matmul(A,V) #计算投影，[B, H, T, Dh]
    O=O.permute(0, 2, 1, 3).contiguous().view(B,T,D)#这一步是合并多头，从 [B, H, T, Dh] 还原回 [B, T, D]。.contiguous() 的核心作用是确保张量在内存中以连续（contiguous）的方式存储

    O = torch.matmul(O, Wo) #多头特有的部分，融合各个头的信息。

    '''
    print(Q)
    print(K)
    print(V)
    print(S)
    print(M)
    print(A)
    '''
    return O

In [44]:
gamma=torch.ones(D, requires_grad=True)
beta=torch.zeros(D, requires_grad=True)

def LayerNorm(X): #层归一化
    
    mu=torch.mean(X,dim=-1,keepdims=True)
    sigma=torch.var(X,axis=-1,keepdims=True)
    normalized_X=(X-mu)/torch.sqrt(sigma+1e-5)
    return gamma*normalized_X+beta #数值乘法和加法，矩阵乘法是torch.matmul。用了gamma和beta这个Loss从0.3726干到0.0643了，推测是不加beta一堆负值过不了gelu。如果模型觉得 “不归一化更好”，它可以通过学习近似恢复出归一化前的原始特征。

In [45]:
W1=torch.randn(D,4*D,requires_grad=True)
b1=torch.randn(4*D,requires_grad=True)
W2=torch.randn(4*D,D,requires_grad=True)
b2=torch.randn(D,requires_grad=True)

torch.nn.init.normal_(W1, mean=0.0, std=0.02)
torch.nn.init.normal_(b1, mean=0.0, std=0.02)
torch.nn.init.normal_(W2, mean=0.0, std=0.02)
torch.nn.init.normal_(b2, mean=0.0, std=0.02)

def FeedForward(X):
    X=torch.matmul(X,W1)+b1
    X=F.gelu(X)
    X=torch.matmul(X,W2)+b2
    return X

In [46]:
def TransformerBlock(X): #输入 / 输出：[B, T, D]。问：归一化之后token和token之间的相对大小不就变了吗，还怎么区分token和token？答：归一化之后token向量方向不变，长度变了。长度是一维信息，在方向里多加一维就效果一样了。
    X = X + MultiHeadCausalSelfAttention(LayerNorm(X))
    X = X + FeedForward(LayerNorm(X))
    return X

In [47]:
def FinalLayerNorm(X): #就是普通的 LayerNorm
    return LayerNorm(X)

In [48]:
W_head=torch.randn(D,V,requires_grad=True)
b_head=torch.randn(V,requires_grad=True)

torch.nn.init.normal_(W_head, mean=0.0, std=0.02)
torch.nn.init.normal_(b_head, mean=0.0, std=0.02)

def LinearHead(X): #线性层，输入 [B, T, D] → 输出 [B, T, V]，参数 W_head [D, V], b_head [V]。
    X=torch.matmul(X,W_head)+b_head
    return X

In [49]:
def miniGPT(token_ids): #设计的时候要同时能适用于[B,T]和[T]
    X = TokenEmbedding(token_ids) 
    
    X = LearnablePositionalEncoding(X)

    X = TransformerBlock(X) #先只叠一个TransformerBlock

    X = FinalLayerNorm(X)

    logits = LinearHead(X)

    return logits

In [50]:
all_params = [
    embedding_matrix, pos_embedding_matrix,
    Wq, Wk, Wv, Wo,
    gamma,beta,
    W1, b1, W2, b2,
    W_head, b_head
]

In [51]:
def train(train_text):
    lr=1e-3
    epochs=1000
    seq_len=4

    optimizer=torch.optim.AdamW(all_params,lr=lr) #优化器
    

    X_train=[]
    Y_train=[]
    for i in range(len(train_text)-seq_len):
        X_train.append([char_to_idx[c] for c in train_text[i:i+seq_len]])
        Y_train.append([char_to_idx[c] for c in train_text[i+1:i+seq_len+1]])

    X_train=torch.tensor(X_train)
    Y_train=torch.tensor(Y_train)

    for epoch in range(epochs):
        logits=miniGPT(X_train)
        #logits需要展平为[B*T, V]，Y展平为[B*T]
        loss=F.cross_entropy(logits.reshape(-1,V), Y_train.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch%50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [52]:
def generate(prompt_str,max_new_tokens=10):
    input_ids=torch.tensor([[char_to_idx[c] for c in prompt_str]]) #input_ids是[B, T]

    for _ in range(max_new_tokens):
        logits=miniGPT(input_ids)
        next_id=torch.argmax(logits[:,-1,:],dim=-1,keepdim=True) #logits[:,-1,:]是每个batch最后一个token的概率分布

        input_ids=torch.cat((input_ids,next_id),dim=-1)

    res=[''.join([idx_to_char[i.item()] for i in input_ids[0,:]])]
    return res

In [53]:
if __name__ == "__main__":
    train("abcabcabcabcabcabcabcabcab")
    print("生成结果：", generate("a", max_new_tokens=10))

Epoch 0, Loss: 3.2443
Epoch 50, Loss: 2.8093
Epoch 100, Loss: 2.3384
Epoch 150, Loss: 1.9046
Epoch 200, Loss: 1.5871
Epoch 250, Loss: 1.3958
Epoch 300, Loss: 1.2871
Epoch 350, Loss: 1.2080
Epoch 400, Loss: 1.0901
Epoch 450, Loss: 0.9240
Epoch 500, Loss: 0.7296
Epoch 550, Loss: 0.5376
Epoch 600, Loss: 0.3867
Epoch 650, Loss: 0.2814
Epoch 700, Loss: 0.2104
Epoch 750, Loss: 0.1621
Epoch 800, Loss: 0.1284
Epoch 850, Loss: 0.1042
Epoch 900, Loss: 0.0862
Epoch 950, Loss: 0.0726
生成结果： ['abcabcabcab']
